# Ch 13 — Mixed Precision과 Grad Accumulation

bf16/fp16 `torch.autocast` 한 줄로 mixed precision 을 켜고, gradient accumulation 으로 작은 GPU 에서 큰 effective batch 를 흉내낸다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part4/ch13_mixed_precision.ipynb)

In [ ]:
# !pip install -q torch tokenizers datasets

import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from dataclasses import dataclass

# Device check
if torch.cuda.is_available():
    device = 'cuda'
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'device: {device}')
print(f'torch:  {torch.__version__}')

## GPTMini 모델 정의

Ch 12 와 동일한 모델 구조.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 8000
    n_layer: int = 6
    n_head: int = 8
    d_model: int = 320
    max_len: int = 512

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.d_model = cfg.d_model
        self.qkv  = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))

class FFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        h = int(cfg.d_model * 8 / 3)
        self.gate = nn.Linear(cfg.d_model, h, bias=False)
        self.up   = nn.Linear(cfg.d_model, h, bias=False)
        self.down = nn.Linear(h, cfg.d_model, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.RMSNorm(cfg.d_model)
        self.attn  = CausalSelfAttention(cfg)
        self.norm2 = nn.RMSNorm(cfg.d_model)
        self.ffn   = FFN(cfg)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class GPTMini(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg    = cfg
        self.embed  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos    = nn.Embedding(cfg.max_len, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm   = nn.RMSNorm(cfg.d_model)
        self.head   = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, x, y=None):
        B, T = x.shape
        h = self.embed(x) + self.pos(torch.arange(T, device=x.device))
        for block in self.blocks:
            h = block(h)
        logits = self.head(self.norm(h))
        loss = None
        if y is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg.vocab_size), y.view(-1))
        return logits, loss

cfg = GPTConfig()
print(f'모델 파라미터: {sum(p.numel() for p in GPTMini(cfg).parameters())/1e6:.1f}M')

def get_batch(batch_size=8, seq_len=128, vocab_size=8000):
    x = torch.randint(0, vocab_size, (batch_size, seq_len))
    y = torch.randint(0, vocab_size, (batch_size, seq_len))
    return x.to(device), y.to(device)

## 최소 예제 — autocast 한 줄

PyTorch 가 mixed precision 을 자동 처리. `autocast` 안의 forward 가 자동으로 bf16/fp16.

In [ ]:
# bf16 (A100/H100/M2 권장)
model = GPTMini(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=6e-4, betas=(0.9, 0.95))
scaler = GradScaler()   # fp16 면 필요, bf16 면 불필요

# 실제 dtype 결정
if device == 'cuda' and torch.cuda.is_bf16_supported():
    amp_dtype = torch.bfloat16
    use_scaler = False
elif device == 'cuda':
    amp_dtype = torch.float16
    use_scaler = True
else:
    amp_dtype = torch.float32
    use_scaler = False

print(f'amp_dtype:  {amp_dtype}')
print(f'use_scaler: {use_scaler}')

In [ ]:
# bf16 학습 루프 (A100/H100 권장)
model.train()
x, y = get_batch()

# bf16 (A100/H100 권장)
with autocast(device_type=device if device != 'mps' else 'cpu',
              dtype=amp_dtype,
              enabled=(device == 'cuda')):
    logits, loss = model(x, y)

loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
optimizer.zero_grad(set_to_none=True)

print(f'loss: {loss.item():.3f}')
print(f'logits dtype: {logits.dtype}')

In [ ]:
# T4 (fp16) 버전 — GradScaler 필요
def train_step_fp16(model, optimizer, scaler, x, y):
    with autocast(device_type='cuda', dtype=torch.float16):
        logits, loss = model(x, y)

    scaler.scale(loss).backward()         # loss 에 큰 수 곱해 backward. underflow 방지.
    scaler.unscale_(optimizer)            # clip 전에 unscale
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    return loss.item()

print('fp16 + GradScaler 패턴 정의 완료')
print('GradScaler 현재 scale:', scaler.get_scale() if use_scaler else 'N/A (bf16)')

## Gradient Accumulation — 큰 batch 흉내

**N 번 forward+backward 하면서 gradient 누적** → N 번째에 한 번 step.  
effective batch = batch × N.

In [ ]:
accum_steps = 4   # effective batch = 8 * 4 = 32

model2 = GPTMini(cfg).to(device)
opt2   = torch.optim.AdamW(model2.parameters(), lr=6e-4, betas=(0.9, 0.95))

model2.train()
opt2.zero_grad(set_to_none=True)

for step in range(20):
    x, y = get_batch()

    with autocast(device_type=device if device != 'mps' else 'cpu',
                  dtype=amp_dtype,
                  enabled=(device == 'cuda')):
        logits, loss = model2(x, y)
        loss = loss / accum_steps                                       # N 으로 나눔 — 평균

    loss.backward()                                                    # gradient 누적

    if (step + 1) % accum_steps == 0:                                  # N 번째에 step
        torch.nn.utils.clip_grad_norm_(model2.parameters(), 1.0)
        opt2.step()
        opt2.zero_grad(set_to_none=True)
        print(f'  optimizer step at step {step+1}, loss ≈ {loss.item()*accum_steps:.3f}')

## Gradient Checkpointing — activation 메모리 절약

forward 의 중간 activation 을 저장하지 않고 layer 경계만 저장.  
backward 할 때 재계산 → 메모리 1/√L, 시간 1.3×.

In [ ]:
from torch.utils.checkpoint import checkpoint

class BlockWithCheckpoint(nn.Module):
    """gradient checkpointing 적용된 Block."""
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.RMSNorm(cfg.d_model)
        self.attn  = CausalSelfAttention(cfg)
        self.norm2 = nn.RMSNorm(cfg.d_model)
        self.ffn   = FFN(cfg)

    def forward(self, x):
        # 원래: x = x + self.attn(self.norm1(x))
        # 체크포인팅 적용
        x = x + checkpoint(self.attn, self.norm1(x), use_reentrant=False)
        x = x + checkpoint(self.ffn, self.norm2(x), use_reentrant=False)
        return x

print('BlockWithCheckpoint 정의 완료')
print('본 책 10M 모델은 이게 필요 없음 (메모리 여유).')
print('30M+ 또는 큰 batch 쓸 때 적용.')

## 실전 — precision 별 속도·메모리 비교

In [ ]:
# precision 별 1 step 시간 비교 (CUDA 있을 때만 의미 있음)
import gc

def measure_step(dtype_str, n_steps=50):
    m = GPTMini(cfg).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=6e-4)
    m.train()

    if dtype_str == 'fp32':
        amp_ctx = torch.no_grad.__class__  # dummy
        use_ctx = False
    else:
        dt = torch.bfloat16 if dtype_str == 'bf16' else torch.float16
        use_ctx = True

    # warm-up
    x, y = get_batch()
    if use_ctx:
        with autocast(device_type=device if device != 'mps' else 'cpu',
                      dtype=dt, enabled=(device == 'cuda')):
            _, loss = m(x, y)
    else:
        _, loss = m(x, y)
    loss.backward(); opt.step(); opt.zero_grad()

    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_steps):
        x, y = get_batch()
        if use_ctx:
            with autocast(device_type=device if device != 'mps' else 'cpu',
                          dtype=dt, enabled=(device == 'cuda')):
                _, loss = m(x, y)
        else:
            _, loss = m(x, y)
        loss.backward(); opt.step(); opt.zero_grad()
    if device == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.time() - t0

    del m, opt
    if device == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return elapsed / n_steps * 1000  # ms per step

print('측정 중... (각 precision 50 step)')
results = {}
for dtype_str in ['fp32', 'bf16']:
    ms = measure_step(dtype_str)
    results[dtype_str] = ms
    print(f'  {dtype_str}: {ms:.1f} ms/step')

if 'fp32' in results and 'bf16' in results:
    speedup = results['fp32'] / results['bf16']
    print(f'\nbf16 speedup: {speedup:.1f}x (GPU 종류에 따라 다름)')

## 연습문제

1. 본인 GPU 에서 같은 학습을 `fp32` / `bf16` / `fp16` 세 가지로 1000 step 돌려 (a) 시간 (b) 메모리 (c) loss 비교.
2. `accum_steps` 를 1 / 4 / 16 로 바꿔 effective batch 를 같게 (예: 128) 유지하면서 시간·메모리·최종 loss 비교.
3. 30M 모델에 gradient checkpointing 적용 vs 미적용으로 메모리·시간 측정. 1.3× 시간 비용이 1/√L 메모리 절약을 정당화하는가?
4. fp16 학습 중 일부러 큰 lr (`3e-3`) 을 줘서 overflow → NaN 을 발생시켜라. `GradScaler` 가 어떻게 동작하는지 (`scaler.get_scale()` 출력) 관찰.
5. **(생각해볼 것)** bf16 이 fp32 와 같은 범위라면 왜 fp32 학습이 아직도 가끔 필요한가? 어느 부분에서 정밀도가 문제 되는가?

In [ ]:
# 연습 1: fp32 / bf16 / fp16 비교


In [ ]:
# 연습 2: accum_steps 실험


In [ ]:
# 연습 3: gradient checkpointing 메모리·시간 측정


In [ ]:
# 연습 4: fp16 overflow 관찰


In [ ]:
# 연습 5: 자유 답변
